In [ ]:
# from glob import glob
import os.path as op

# import warnings
# from IPython.display import HTML

# import math
# from math import ceil
# import matplotlib as mpl
# import matplotlib.gridspec as gridspec
# import matplotlib.pyplot as plt
# import numpy.typing as npt
import numpy as np
import nibabel as nb
# from numpy import linalg, polyfit

# from nilearn.image import check_niimg, new_img_like
import pandas as pd
# from scipy.stats import zscore, linregress
# from scipy.optimize import curve_fit
# from sklearn.decomposition import PCA, FastICA

# from tedana.stats import get_coeffs

# from tedana import io, stats, utils
# from tedana.utils import get_spectrum

import sys

sys.path.append("/Users/tkmd/Projects/multiecho-simulation/")

# from utils_tt import file_to_2d, file_to_2d, mask_data, \
#     plot_time_and_echoes_brain, plot_echo_profiles,compare_rsqared, \
#     pca_dim_reduced, get_new_slopes, initialize_cp_with_ica, parafac_iter, \
#         ica_to_parafac_iter


# from generate_sims import generate_sims as gen_sims
import simlib.generate_sims as gen_sims

# Run tedana on a dataset with X components (X=30?)

Generate OC data! Done for subj 1----


```bash

tedana -d p05.SBJ01_S09_Task11_e1.in.nii.gz \
           p05.SBJ01_S09_Task11_e2.in.nii.gz \
           p05.SBJ01_S09_Task11_e3.in.nii.gz \
           p05.SBJ01_S09_Task11_e4.in.nii.gz \
           p05.SBJ01_S09_Task11_e5.in.nii.gz \
         --mask mask.nii.gz \
         -e 15.4 29.7 44.0 58.3 72.6 \
         --tree minimal \
         --tedpca 30 \
         --out-dir tedana_out_for_sim_s09_t11 --overwrite
```


In [29]:
# Load the ICA mixing matrix (desc-ICA_mixing.tsv), the ICA weight maps
# (desc-ICA_components.nii.gz) and the component metrics table
# (desc-tedana_metrics.tsv)

data_length = 340

drive_loc = "/Users/tkmd/Projects/multiecho-simulation"

mixing_file = op.join(drive_loc, 'tedana_test/desc-ICA_mixing.tsv')
comps_file = op.join(drive_loc, 'tedana_test/desc-ICA_components.nii.gz')
metrics_file = op.join(drive_loc, 'tedana_test/desc-tedana_metrics.tsv')

mixing = np.loadtxt(mixing_file, delimiter='\t', skiprows=1)
comps_img = nb.load(comps_file).get_fdata()
x, y, z, a = comps_img.shape
comps = comps_img.reshape(x*y*z, a).astype(int)

metrics_df = pd.read_csv(metrics_file, delimiter = "\t")

print(mixing.shape)
print(comps.shape)

(340, 38)
(871200, 38)


In [ ]:
# For each component, proportion_s0_r2s=1 if the “classification” column is
# “rejected” and 0 of “accepted"
n_comps = a

proportion_s0_r2s_vec = np.zeros(n_comps)
print(proportion_s0_r2s_vec)
for component in range(n_comps):
    if metrics_df["classification"][component] == "accepted":
        proportion_s0_r2s_vec[component] = 1

print(proportion_s0_r2s_vec)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 1. 1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 1. 1.]


In [23]:
max = np.max(mixing)
print(max)

inputted_s = mixing/max*0.1
s0_mean=500
r2s_mean = 1/40
te_baseline=1/30

proportion_s0_r2s = np.tile(proportion_s0_r2s_vec, (inputted_s.shape[0], 1))
print(proportion_s0_r2s)
print(proportion_s0_r2s.shape)

8.8870884037552
[[0. 1. 0. ... 1. 1. 1.]
 [0. 1. 0. ... 1. 1. 1.]
 [0. 1. 0. ... 1. 1. 1.]
 ...
 [0. 1. 0. ... 1. 1. 1.]
 [0. 1. 0. ... 1. 1. 1.]
 [0. 1. 0. ... 1. 1. 1.]]
(340, 38)


In [ ]:

print(inputted_s.shape, proportion_s0_r2s.shape)

delta_s0, delta_r2s = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
        inputted_s=inputted_s,
        s0_mean=s0_mean,
        r2s_mean=r2s_mean,
        proportion_s0_r2s=proportion_s0_r2s,
        te_baseline=te_baseline)

data_shape = x, y, z, data_length
tes = [10, 20, 30, 40, 50]

for te in tes:

    print(te)
    new_mixing = gen_sims.monoexponential(te, s0_mean + delta_s0,
                                          r2s_mean + delta_r2s)

    # print(np.sum(new_mixing), np.sum(comps))
    # print(new_mixing.shape, comps.shape)
    new_data = np.dot(comps, new_mixing.T)
    # print(np.sum(new_data))
    # print(new_data.shape)

    print(x, y, z)
    array = np.reshape(new_data, (x, y, z, data_length))

    print(array.shape)
    new_image = nb.Nifti1Image(array, np.eye(4))

    nb.save(new_image, op.join(f"~/Projects/multiecho-simulation/data_echo_time_{te}.nii.gz"))



(340, 38) (340, 38)
Before var_shapes=[(1,), (1,), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
10
Before var_shapes=[(1,), (340, 38), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
110 110 72
(110, 110, 72, 340)
20
Before var_shapes=[(1,), (340, 38), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
110 110 72
(110, 110, 72, 340)
30
Before var_shapes=[(1,), (340, 38), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
110 110 72
(110, 110, 72, 340)
40
Before var_shapes=[(1,), (340, 38), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
110 110 72
(110, 110, 72, 340)
50
Before var_shapes=[(1,), (340, 38), (340, 38)]
After var_shapes=[(340, 38), (340, 38), (340, 38)]
110 110 72
(110, 110, 72, 340)


In [36]:
comps = comps_img.reshape(x*y*z, a).astype(int)
validate = np.reshape(comps, (x, y, z, a))

new_image = nb.Nifti1Image(validate, np.eye(4), dtype = np.int16)

nb.save(new_image, op.join("~/Projects/multiecho-simulation/",
                            "validate.nii.gz"))


How to recapitulate tedana:

```bash
tedana -d data_echo_time_10.nii \
          data_echo_time_20.nii \
          data_echo_time_30.nii \
          data_echo_time_40.nii \
          data_echo_time_50.nii \
         --mask mask.nii.gz \
         -e 10 20 30 40 50 \
         --tree minimal \
         --tedpca 10 \
         --out-dir tedana_out_for_sim_results --overwrite
```